# ONS API Setup
This notebook sets up a small helper layer for the Office for National Statistics API and pulls a worked UK CPIH time series example.

The flow is:
- define a reusable JSON request helper
- resolve the latest published dataset version
- request wildcard observations for a chosen geography and aggregate
- normalize the response into a tidy pandas DataFrame and plot it

In [1]:
from __future__ import annotations

import json
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import pandas as pd
import plotly.express as px

ONS_API_BASE = "https://api.beta.ons.gov.uk/v1"
DEFAULT_HEADERS = {"User-Agent": "FinanceNotebook/1.0"}


def ons_get_json(path: str, *, params: dict[str, str] | None = None) -> dict:
    query = f"?{urlencode(params)}" if params else ""
    request = Request(f"{ONS_API_BASE}{path}{query}", headers=DEFAULT_HEADERS)
    with urlopen(request, timeout=30) as response:
        return json.load(response)


def get_latest_dataset_version(dataset_id: str, *, edition: str = "time-series") -> str:
    payload = ons_get_json(f"/datasets/{dataset_id}")
    href = payload["links"]["latest_version"]["href"]
    return href.rstrip("/").split("/")[-1]


def get_dimension_options(
    dataset_id: str,
    dimension_name: str,
    *,
    edition: str = "time-series",
    version: str | None = None,
) -> pd.DataFrame:
    resolved_version = version or get_latest_dataset_version(dataset_id, edition=edition)
    payload = ons_get_json(
        f"/datasets/{dataset_id}/editions/{edition}/versions/{resolved_version}/dimensions/{dimension_name}/options"
    )
    return pd.DataFrame(payload.get("items", []))


def fetch_dataset_observations(
    dataset_id: str,
    *,
    edition: str = "time-series",
    version: str | None = None,
    dimensions: dict[str, str],
) -> dict:
    resolved_version = version or get_latest_dataset_version(dataset_id, edition=edition)
    return ons_get_json(
        f"/datasets/{dataset_id}/editions/{edition}/versions/{resolved_version}/observations",
        params=dimensions,
    )


def normalize_observations(payload: dict) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    fixed_dimensions = payload.get("dimensions", {})

    for item in payload.get("observations") or []:
        row: dict[str, object] = {}

        for dimension_name, detail in fixed_dimensions.items():
            option = detail.get("option", {})
            row[dimension_name.lower()] = option.get("id")
            row[f"{dimension_name.lower()}_label"] = option.get("label", option.get("id"))

        for dimension_name, detail in item.get("dimensions", {}).items():
            key = dimension_name.lower()
            row[key] = detail.get("id")
            row[f"{key}_label"] = detail.get("label", detail.get("id"))

        row["value"] = pd.to_numeric(item.get("observation"), errors="coerce")
        rows.append(row)

    frame = pd.DataFrame(rows)
    if frame.empty:
        return frame

    if "time" in frame.columns:
        frame["observation_date"] = pd.to_datetime(frame["time"], format="%b-%y", errors="coerce")
        frame = frame.sort_values("observation_date").reset_index(drop=True)

    unit = payload.get("unit_of_measure")
    if unit is not None:
        frame["unit_of_measure"] = unit

    return frame

## Worked Example: UK CPIH
This example uses the ONS CPIH dataset `cpih01` and requests all available time observations for the UK-wide geography and the headline aggregate code used in the ONS documentation example. The notebook resolves the latest published version dynamically, so you do not need to hard-code version numbers.

In [2]:
DATASET_ID = "cpih01"

latest_version = get_latest_dataset_version(DATASET_ID)
aggregate_options = get_dimension_options(DATASET_ID, "aggregate", version=latest_version)
geography_options = get_dimension_options(DATASET_ID, "geography", version=latest_version)

GEOGRAPHY_ID = geography_options.loc[0, "option"]
AGGREGATE_ID = "CP00"

cpih_payload = fetch_dataset_observations(
    DATASET_ID,
    version=latest_version,
    dimensions={
        "time": "*",
        "geography": GEOGRAPHY_ID,
        "aggregate": AGGREGATE_ID,
    },
)

cpih_frame = normalize_observations(cpih_payload)

print(f"Latest version: {latest_version}")
display(aggregate_options.loc[:, ["option", "label"]].head())
cpih_frame.tail()

Latest version: 67


,option,label
0,CP00,Overall Index
1,CP01,01 Food and non-alcoholic beverages
2,CP02,02 Alcoholic beverages and tobacco
3,CP03,03 Clothing and footwear
4,CP04,"04 Housing, water, electricity, gas and other ..."


,aggregate,aggregate_label,geography,geography_label,time,time_label,value,observation_date,unit_of_measure
452,CP00,CP00,K02000001,K02000001,Sep-25,Sep-25,138.9,2025-09-01,Index: 2015=100
453,CP00,CP00,K02000001,K02000001,Oct-25,Oct-25,139.5,2025-10-01,Index: 2015=100
454,CP00,CP00,K02000001,K02000001,Nov-25,Nov-25,139.4,2025-11-01,Index: 2015=100
455,CP00,CP00,K02000001,K02000001,Dec-25,Dec-25,139.9,2025-12-01,Index: 2015=100
456,CP00,CP00,K02000001,K02000001,Jan-26,Jan-26,139.4,2026-01-01,Index: 2015=100


In [3]:
cpih_summary = cpih_frame.loc[:, ["observation_date", "value", "unit_of_measure"]].copy()
cpih_summary = cpih_summary.rename(columns={"value": "cpih_index"})

cpih_chart = px.line(
    cpih_summary,
    x="observation_date",
    y="cpih_index",
    title=f"ONS CPIH Time Series ({DATASET_ID}, version {latest_version})",
    labels={
        "observation_date": "Observation date",
        "cpih_index": cpih_payload.get("unit_of_measure", "Value"),
    },
)
cpih_chart.update_layout(height=480)
cpih_chart.show()

cpih_summary.describe(include="all")

,observation_date,cpih_index,unit_of_measure
count,457,457.000000,457
unique,NaN,NaN,1
top,NaN,NaN,Index: 2015=100
freq,NaN,NaN,457
mean,2006-12-31 10:33:20.875273,86.569764,NaN
min,1988-01-01 00:00:00,46.900000,NaN
25%,1997-07-01 00:00:00,70.100000,NaN
50%,2007-01-01 00:00:00,82.400000,NaN
75%,2016-07-01 00:00:00,101.000000,NaN
max,2026-01-01 00:00:00,139.900000,NaN


## Next Adaptations
To reuse this notebook for another ONS series, keep the helper functions and change only these values in Cell 4:
- `DATASET_ID`
- the dimension keys and option IDs passed in `dimensions`
- any downstream column renaming for the final chart

For more complex queries with multiple wildcards or exported CSV outputs, the next step would be the ONS filter workflow rather than the direct observations endpoint.

In [4]:
# CPIH headline breakdown by top-level divisions

top_level_breakdown_options = aggregate_options.loc[
    aggregate_options["option"].str.fullmatch(r"CP\d{2}") & (aggregate_options["option"] != "CP00"),
    ["option", "label"],
].copy()
top_level_breakdown_options["clean_label"] = top_level_breakdown_options["label"].str.replace(
    r"^\d{2}\s+", "", regex=True
)

breakdown_frames: list[pd.DataFrame] = []
for aggregate_code, aggregate_label, clean_label in top_level_breakdown_options.itertuples(index=False):
    breakdown_payload = fetch_dataset_observations(
        DATASET_ID,
        version=latest_version,
        dimensions={
            "time": "*",
            "geography": GEOGRAPHY_ID,
            "aggregate": aggregate_code,
        },
    )
    breakdown_frame = normalize_observations(breakdown_payload)
    breakdown_frame["aggregate"] = aggregate_code
    breakdown_frame["aggregate_label"] = clean_label
    breakdown_frames.append(
        breakdown_frame.loc[:, ["observation_date", "value", "aggregate", "aggregate_label"]]
    )

cpih_breakdown = pd.concat(breakdown_frames, ignore_index=True)
cpih_breakdown["yoy_change_pct"] = cpih_breakdown.groupby("aggregate")["value"].pct_change(12).mul(100)

latest_breakdown = (
    cpih_breakdown.dropna(subset=["yoy_change_pct"])
    .sort_values("observation_date")
    .groupby("aggregate", as_index=False)
    .tail(1)
    .sort_values("yoy_change_pct", ascending=True)
)

latest_breakdown_chart = px.bar(
    latest_breakdown,
    x="yoy_change_pct",
    y="aggregate_label",
    orientation="h",
    color="yoy_change_pct",
    color_continuous_scale="RdYlGn_r",
    title=f"Latest CPIH Annual Change by Division ({latest_breakdown['observation_date'].max():%b %Y})",
    labels={
        "aggregate_label": "Division",
        "yoy_change_pct": "Annual change (%)",
    },
)
latest_breakdown_chart.update_layout(height=520, coloraxis_showscale=False)
latest_breakdown_chart.show()

recent_breakdown = cpih_breakdown.loc[
    cpih_breakdown["observation_date"] >= cpih_breakdown["observation_date"].max() - pd.DateOffset(years=3)
].copy()
recent_breakdown = recent_breakdown.dropna(subset=["yoy_change_pct"])

recent_breakdown_chart = px.line(
    recent_breakdown,
    x="observation_date",
    y="yoy_change_pct",
    color="aggregate_label",
    title="CPIH Division Breakdown: Annual Change Over the Last 3 Years",
    labels={
        "observation_date": "Observation date",
        "yoy_change_pct": "Annual change (%)",
        "aggregate_label": "Division",
    },
)
recent_breakdown_chart.update_layout(height=560)
recent_breakdown_chart.show()

latest_breakdown.loc[:, ["aggregate_label", "observation_date", "yoy_change_pct"]].reset_index(drop=True)

,aggregate_label,observation_date,yoy_change_pct
0,"Furniture, household equipment and maintenance",2026-01-01,-0.479616
1,Clothing and footwear,2026-01-01,0.000000
2,Miscellaneous goods and services,2026-01-01,2.220395
3,Recreation and culture,2026-01-01,2.651515
4,Transport,2026-01-01,2.704733
5,Health,2026-01-01,3.115942
6,Food and non-alcoholic beverages,2026-01-01,3.515065
7,Restaurants and hotels,2026-01-01,4.074586
8,"Housing, water, electricity, gas and other fuels",2026-01-01,4.233577
9,Communication,2026-01-01,4.562178
